<a href="https://colab.research.google.com/github/EhsanGhasemi423/INFS-8368/blob/main/RNN_P2_Converting_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Converting Raw Text into Sequence Data

So far, we have discussed sequences mathematically. Now we will see how **raw text is converted into a numerical sequence that a neural network can process**.

For example, a computer cannot directly use:

`"the time machine"`

Instead, we will eventually convert the words into numerical IDs, such as:

`[1, 25, 73]`

The preprocessing pipeline is:

**Raw text → Clean text → Tokens → Vocabulary → Numerical sequence**

1. **Read the text** — load the raw text as a string.
2. **Preprocess** — remove unwanted characters and convert text to lowercase.
3. **Tokenize** — split the text into individual words or characters.
4. **Build a vocabulary** — assign each unique token a numerical ID.
5. **Convert to indices** — replace each token with its numerical ID.

> **Key idea:** Neural networks work with numbers, not raw words.  
> This section prepares text so that it can later be used as sequence data by an RNN.

### Reading and Cleaning the Text

We begin with *The Time Machine* by H. G. Wells.

The text is first loaded as one long string and then cleaned by:

- converting uppercase letters to lowercase;
- removing punctuation and other non-letter characters.

For example:

`"The Time Machine, by H. G. Wells!"`

becomes

`"the time machine by h g wells"`

This gives us clean text that can next be **tokenized**.

In [ ]:
!pip install --q d2l  --no-deps


In [ ]:
import collections
import random
import re
import tensorflow as tf
from d2l import tensorflow as d2l


In [ ]:
# Define a dataset class for The Time Machine.
class TimeMachine(d2l.DataModule):
    """The Time Machine dataset."""

    # Download and read the book.
    def _download(self):

        # Download the text file.
        fname = d2l.download(
            d2l.DATA_URL + 'timemachine.txt',
            self.root,
            '090b5e7e70c295757f55df93cb0a180b9691891a'
        )

        # Read the entire book as one string.
        with open(fname) as f:
            return f.read()


# Create the dataset.
data = TimeMachine()

# Load the raw text.
raw_text = data._download()

# Display the first 60 characters.
print(raw_text[:60])

The Time Machine, by H. G. Wells [1898]




I


The Time Tra


## Tokenization

After cleaning the raw text, the next step is to **tokenize** it.

A **token** is the basic unit of text that the model processes. A token can be a word, part of a word, or a character.

In this example, we use **character-level tokenization**, meaning that **each individual character becomes one token**.

For example:

$$
\text{"time"} \rightarrow [\text{'t'},\text{'i'},\text{'m'},\text{'e'}]
$$

Spaces are also characters, so they are kept as tokens.

The code uses `list(text)` to split the cleaned text into individual characters.

> **Key idea:** Tokenization converts one long string of text into a sequence of smaller units. At this point, the tokens are still characters, not numbers.

The next step is to build a **vocabulary**, which will assign each unique character a numerical ID.

**Pipeline so far:**

**Raw text → Clean text → Character tokens → Vocabulary → Numerical IDs**

In [ ]:
# Add a tokenization method to the TimeMachine class.
@d2l.add_to_class(TimeMachine)
def _tokenize(self, text):
    # Split the text into individual characters.
    # Each character becomes one token.
    return list(text)


# Tokenize the cleaned text.
tokens = data._tokenize(text)

# Display the first 30 tokens separated by commas.
','.join(tokens[:30])

't,h,e, ,t,i,m,e, ,m,a,c,h,i,n,e, ,b,y, ,h, ,g, ,w,e,l,l,s, '

## Vocabulary

After tokenization, our text has been split into tokens, but the tokens are still **characters** such as:

`['t', 'h', 'e', ' ', 't', 'i', 'm', 'e']`

A neural network cannot work directly with these characters. We therefore create a **vocabulary** that assigns each unique token a numerical ID.

For example:

| Token | Numerical ID |
|---|---:|
| `<unk>` | 0 |
| `a` | 1 |
| `b` | 2 |
| `e` | 3 |
| `h` | 4 |
| `t` | 5 |

The exact IDs depend on how the vocabulary is constructed.

The vocabulary allows us to convert:

**tokens → numerical IDs**

For example:

`['t', 'h', 'e'] → [5, 4, 3]`

It can also perform the reverse conversion:

`[5, 4, 3] → ['t', 'h', 'e']`

The special token **`<unk>` (unknown)** is used when we encounter a token that is not in the vocabulary.

> **Key idea:** The vocabulary creates the mapping between text tokens and the numbers that a neural network can process.

**Pipeline so far:**

**Raw text → Clean text → Tokens → Vocabulary → Numerical sequence**

In [ ]:
# Define a vocabulary class.
# The vocabulary maps tokens <-> numerical IDs.
class Vocab:

    def __init__(self, tokens=[], min_freq=0, reserved_tokens=[]):

        # If tokens are stored as a list of lists,
        # combine them into one list.
        if tokens and isinstance(tokens[0], list):
            tokens = [
                token
                for line in tokens
                for token in line
            ]

        # Count how many times each token appears.
        counter = collections.Counter(tokens)

        # Sort tokens from most frequent to least frequent.
        self.token_freqs = sorted(
            counter.items(),
            key=lambda x: x[1],
            reverse=True
        )

        # Create the list of unique tokens.
        # <unk> represents an unknown token.
        self.idx_to_token = list(
            sorted(
                set(
                    ['<unk>'] +
                    reserved_tokens +
                    [
                        token
                        for token, freq in self.token_freqs
                        if freq >= min_freq
                    ]
                )
            )
        )

        # Create the reverse mapping:
        # token -> numerical ID
        self.token_to_idx = {
            token: idx
            for idx, token in enumerate(self.idx_to_token)
        }


    # Return the number of tokens in the vocabulary.
    def __len__(self):
        return len(self.idx_to_token)


    # Convert token(s) -> numerical ID(s).
    def __getitem__(self, tokens):

        # If only one token is given.
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)

        # If several tokens are given.
        return [self.__getitem__(token) for token in tokens]


    # Convert numerical ID(s) -> token(s).
    def to_tokens(self, indices):

        # If several IDs are given.
        if hasattr(indices, '__len__') and len(indices) > 1:
            return [
                self.idx_to_token[int(index)]
                for index in indices
            ]

        # If only one ID is given.
        return self.idx_to_token[indices]


    # Return the numerical ID used for unknown tokens.
    @property
    def unk(self):
        return self.token_to_idx['<unk>']

In [ ]:
# Build the vocabulary from our character tokens.
vocab = Vocab(tokens)

# Convert the first 10 tokens into numerical IDs.
indices = vocab[tokens[:10]]

print('indices:', indices)

# Convert the numerical IDs back into characters.
print('tokens:', vocab.to_tokens(indices))

indices: [21, 9, 6, 0, 21, 10, 14, 6, 0, 14]
tokens: ['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm']


## Putting It All Together

We now combine the previous preprocessing steps into one pipeline.

Starting with the **raw text**, we:

1. **Preprocess** the text — remove unwanted characters and convert it to lowercase.
2. **Tokenize** the text — split it into individual characters.
3. **Build the vocabulary** — assign each unique character a numerical ID.
4. **Create the corpus** — replace every character in the text with its numerical ID.

The complete process is:

**Raw text → Clean text → Character tokens → Vocabulary → Numerical corpus**

For example:

`"the"` → `['t', 'h', 'e']` → `[21, 9, 6]`

The **corpus** is therefore the entire book represented as a sequence of numbers that can later be used by the neural network.

> **Key idea:**  We are simply combining all the previous preprocessing steps to convert the entire text into a numerical sequence.

In [ ]:
# Add the preprocessing method to the TimeMachine class.
@d2l.add_to_class(TimeMachine)
def _preprocess(self, text):

    # Replace anything that is NOT a letter with a space.
    # Then convert all letters to lowercase.
    return re.sub('[^A-Za-z]+', ' ', text).lower()

In [ ]:
# Add the tokenization method to the TimeMachine class.
@d2l.add_to_class(TimeMachine)
def _tokenize(self, text):

    # Split the text into individual characters.
    return list(text)

In [ ]:
# Combine all preprocessing steps.
@d2l.add_to_class(TimeMachine)
def build(self, raw_text, vocab=None):

    # 1. Clean the raw text.
    # 2. Tokenize it into characters.
    tokens = self._tokenize(self._preprocess(raw_text))

    # 3. Build the vocabulary.
    if vocab is None:
        vocab = Vocab(tokens)

    # 4. Convert each character into its numerical ID.
    corpus = [vocab[token] for token in tokens]

    # Return the numerical corpus and vocabulary.
    return corpus, vocab


# Build the numerical corpus.
corpus, vocab = data.build(raw_text)

# Show the number of tokens and vocabulary size.
len(corpus), len(vocab)

(173428, 28)

## Patterns in Language

Text is not a random sequence of words. Some words and combinations of words occur much more frequently than others.

We can examine:

- **Unigrams** — individual words  
  Example: `"the"`

- **Bigrams** — sequences of two words  
  Example: `"the time"`

- **Trigrams** — sequences of three words  
  Example: `"the time machine"`

These patterns show that **words depend on their surrounding context**.

For example:

`"the time"` → `"machine"`

is much more likely than many random word combinations.

> **Why this matters:** Language contains sequential patterns. A language model attempts to learn these patterns so it can use previous tokens to predict what is likely to come next.

We started with raw text and converted it into a numerical sequence. Now that the text is represented as numbers, we can start building models that learn from those sequences.